# LLama-3.1-KazLLM-1.0-70B — load & generate

Model: [issai/LLama-3.1-KazLLM-1.0-70B](https://huggingface.co/issai/LLama-3.1-KazLLM-1.0-70B) (Llama 3.1 70B fine-tune, Kazakh/Russian/English, BF16, chat template included).

**Before you start**
- The repo is **gated**: log in on Hugging Face and accept the conditions on the model page (Llama 3.1 licence + CC BY-NC 4.0, non-commercial).

In [1]:
%pip install -q -U "transformers>=4.45" accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 41.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.


## 1. Log in to Hugging Face
Use a token from https://huggingface.co/settings/tokens (or set the `HF_TOKEN` environment variable and skip this).

In [ ]:
from huggingface_hub import login
login()  # paste your token when prompted

## 2. Check GPUs

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
total = 0
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    total += p.total_memory / 1024**3
    print(f"GPU {i}: {p.name}  {p.total_memory/1024**3:.0f} GB")
print(f"Total GPU memory: {total:.0f} GB  (BF16 needs ~140 GB, 4-bit ~40 GB)")

CUDA available: True
GPU 0: Tesla T4  15 GB
Total GPU memory: 15 GB  (BF16 needs ~140 GB, 4-bit ~40 GB)


## 3. Load tokenizer and model

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

#MODEL_ID = "issai/LLama-3.1-KazLLM-1.0-70B"
#LOAD_IN_4BIT = False   # True -> fits on a single ~48-80 GB GPU
MODEL_ID = "issai/LLama-3.1-KazLLM-1.0-8B"
LOAD_IN_4BIT = False
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if LOAD_IN_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=quant_config, device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
    )

model.eval()
print(model.hf_device_map if hasattr(model, "hf_device_map") else model.device)

config.json:   0%|          | 0.00/987 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 0, 'model.layers.19': 0, 'model.layers.20': 0, 'model.layers.21': 0, 'model.layers.22': 0, 'model.layers.23': 0, 'model.layers.24': 0, 'model.layers.25': 0, 'model.layers.26': 0, 'model.layers.27': 'cpu', 'model.layers.28': 'cpu', 'model.layers.29': 'cpu', 'model.layers.30': 'cpu', 'model.layers.31': 'cpu', 'model.norm': 'cpu', 'model.rotary_emb': 'cpu', 'lm_head': 'cpu'}


## 4. Generation helper
Uses the model's built-in Llama 3.1 chat template. Llama 3.1 ends turns with `<|eot_id|>`, so it's added as a stop token.

In [6]:
terminators = [tokenizer.eos_token_id]
eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
if eot is not None and eot not in terminators:
    terminators.append(eot)

def chat(messages, max_new_tokens=512, temperature=0.6, top_p=0.9, do_sample=True):
    """messages: list of {"role": "system"|"user"|"assistant", "content": str}"""
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            eos_token_id=terminators,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 5. Try it

In [7]:
messages = [
    {"role": "system", "content": "Сен пайдалы көмекшісің. Қазақ тілінде жауап бер."},
    {"role": "user", "content": "Қазақстанның астанасы туралы қысқаша айтып бер."},
]
print(chat(messages))

KeyboardInterrupt: 

In [ ]:
# English / Russian work too
print(chat([{"role": "user", "content": "Who are you? Answer briefly."}], max_new_tokens=128))

## 6. Streaming output (tokens printed as they're generated)

In [8]:
from transformers import TextStreamer

def chat_stream(messages, max_new_tokens=512, temperature=0.6, top_p=0.9):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    with torch.no_grad():
        model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            eos_token_id=terminators,
            pad_token_id=tokenizer.pad_token_id,
        )

chat_stream([{"role": "user", "content": "Абай Құнанбайұлы кім болған?"}])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Абай Құнанбайұлы (1845-1904) — қазақ ақыны, философ, композитор, қоғам қайраткері, қазақ әдебиеті мен мәдениетінің негізін салушылардың бірі. Ол қазақ халқының мәдениеті мен әдебиетін дамытуға үлкен үлес қосқан. Абай 

KeyboardInterrupt: 

## 7. Multi-turn conversation

In [ ]:
history = [{"role": "system", "content": "Сен пайдалы көмекшісің."}]

def ask(user_text, **kw):
    history.append({"role": "user", "content": user_text})
    reply = chat(history, **kw)
    history.append({"role": "assistant", "content": reply})
    print(reply)

ask("Маған наурыз мейрамы туралы айтып бер.")
ask("Оны қалай тойлайды?")

## 8. Batch generation

In [ ]:
tokenizer.padding_side = "left"  # required for batched decoder-only generation

prompts = [
    "Алматы туралы бір сөйлеммен айт.",
    "Translate to Kazakh: 'Good morning, how are you?'",
    "Что такое бешбармак?",
]
texts = [
    tokenizer.apply_chat_template([{"role": "user", "content": p}], add_generation_prompt=True, tokenize=False)
    for p in prompts
]
batch = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)

with torch.no_grad():
    out = model.generate(
        **batch, max_new_tokens=200, do_sample=False,
        eos_token_id=terminators, pad_token_id=tokenizer.pad_token_id,
    )

for p, o in zip(prompts, out):
    print("Q:", p)
    print("A:", tokenizer.decode(o[batch["input_ids"].shape[-1]:], skip_special_tokens=True).strip(), "\n")

## 9. Free GPU memory (optional)

In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()